In [ ]:
%pip install mne hu-neuro-pipeline pandas

In [ ]:
import pandas as pd
import mne

from mne import (Epochs, events_from_annotations, merge_events,
                 set_bipolar_reference)
from mne.io import read_raw
from mne.viz import plot_compare_evokeds
from mne.preprocessing import ICA
from pipeline.datasets import get_erpcore

raw_clean = mne.io.read_raw_fif(
    "sub-001_run-1_clean_raw.fif",
    preload=True
)

# Group response-tagged sounds by their sound type.
# The dataset contains the spelling "reponse".
annotation_codes = {
    "standard": 1,
    "oddball": 2,
    "oddball_with_reponse": 2,
    "oddball_with_response": 2,
    "noise": 3,
    "noise_with_reponse": 3,
    "noise_with_response": 3,
}

events, _ = mne.events_from_annotations(
    raw_clean, event_id=annotation_codes
)

event_id = {"standard": 1, "oddball": 2, "noise": 3}

epochs = mne.Epochs(
    raw_clean,
    events,
    event_id=event_id,
    tmin=-0.2,
    tmax=0.8,
    baseline=(-0.2, 0),
    picks="eeg",
    preload=True,
    reject_by_annotation=True,
)

print({name: len(epochs[name]) for name in event_id})

In [ ]:
evokeds = {
    condition: epochs[condition].average()
    for condition in ["standard", "oddball", "noise"]
}

# Compare standard and oddball responses at Pz.
_ = mne.viz.plot_compare_evokeds(
    {
        "Standard": evokeds["standard"],
        "Oddball": evokeds["oddball"],
    },
    picks=["Pz"],
    ci=False,
)

In [ ]:
oddball_erp = epochs["oddball"].average()

_ = oddball_erp.plot_topomap(
    times=[0.100, 0.200, 0.350, 0.500],
    ch_type="eeg",
    time_unit="s",
    cmap="RdBu_r",
    colorbar=True,
)